# AMEX Enterprise Credit Risk Platform
## Notebook 20 — Phase 1, Problem 2: Risk Tier Classification — Model Development
### Problem Statement 2 of 14: Risk Tier Classification

CRISP-DM stage: **Modeling / Evaluation**. Second notebook of Problem 2 (Notebooks 19-25). Depends on Notebook 19's `risk_tier_policy.json` (hard dependency) and reuses Notebook 05's real saved champion model and fitted preprocessing artifacts exactly as Notebook 13 does, scoring the same real, held-out test split that was never trained on.

**What this notebook does, all real, live-computed:**

- Loads the real champion model and preprocessing artifacts, scores the real holdout population -- the exact same scoring pattern Notebook 13 uses.
- Computes **both** risk-tier bucketing methods defined in Notebook 19's policy (quantile-based and business-rule/PD-threshold-based) against the real PD scores.
- Measures, for each method, real per-tier population, real per-tier actual bad rate, strict rank-ordering (monotonicity), bad-rate separation ratio, and split-half population stability (PSI) -- and reports the outcome honestly, including if a KPI target is **not** met. A KPI miss is a real finding, not a notebook defect, and is never silently passed.
- Selects the policy's primary method for downstream use, while keeping both methods' assignments in the output for full transparency.

**Deliverables:** `risk_tier_assignments.csv`, tier-performance charts, `Risk_Tier_Model_Development_Report.docx`, `notebook_20_summary.json`.

**Run the single code cell below, once.** Idempotent — every output file is overwritten in place on every re-run.

In [ ]:
# =============================================================================
# SECTION 1: ENVIRONMENT SETUP -- LOAD CONFIG FROM NOTEBOOKS 01, 04, 05, 19
# =============================================================================
import os
import sys
import csv
import json
import time
import warnings
from pathlib import Path
from datetime import datetime, timezone


def _section(title: str) -> None:
    bar = "=" * 78
    print(f"\n{bar}\n{title}\n{bar}")


_section("SECTION 1: Environment Setup -- Load Config From Notebooks 01, 04, 05, 19")

PROJECT_ROOT = Path(r"C:\Users\rnand\Downloads\amex-default-prediction\AMEX_Enterprise_Credit_Risk_Platform")
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
CONFIG_PATH = ARTIFACTS_DIR / "project_config.json"
NB04_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_04_summary.json"
NB05_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_05_summary.json"
NB09_MLOPS_REGISTRY_PATH = None  # resolved after PILLAR_DIRS is known, below

for _p, _fix in [
    (CONFIG_PATH, "run 01_business_understanding.ipynb first"),
    (NB04_SUMMARY_PATH, "run 04_feature_engineering.ipynb first"),
]:
    if not _p.exists():
        raise FileNotFoundError(f"{_p} not found.\nFix: {_fix} -- this notebook reads its outputs.")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PROJECT_CONFIG = json.load(f)
with open(NB04_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB04_SUMMARY = json.load(f)

PILLAR_DIRS = {k: Path(v) for k, v in PROJECT_CONFIG["pillar_dirs"].items()}
RANDOM_SEED = PROJECT_CONFIG["random_seed"]
_resource_limits = PROJECT_CONFIG.get("resource_limits", {})
WARP_THREAD_COUNT = (
    _resource_limits.get("warp_thread_count")
    or PROJECT_CONFIG.get("warp_thread_count")
    or PROJECT_CONFIG["hardware"]["logical_cores_detected"]
)

# --- Self-heal: same pattern as Notebooks 17/18/19 -- a stale config missing
#     Problem 2's pillar keys is fixed in place, no manual re-run required. ---
_REQUIRED_PILLARS = {
    "risk_tier_policy": "02_Problem2_Risk_Tier_Classification/01_Risk_Tier_Policy",
    "risk_tier_modeling": "02_Problem2_Risk_Tier_Classification/02_Risk_Tier_Modeling",
    "risk_tier_validation": "02_Problem2_Risk_Tier_Classification/03_Risk_Tier_Validation",
    "risk_tier_deployment": "02_Problem2_Risk_Tier_Classification/04_Risk_Tier_Deployment",
    "risk_tier_monitoring": "02_Problem2_Risk_Tier_Classification/05_Risk_Tier_Monitoring",
    "risk_tier_reporting": "02_Problem2_Risk_Tier_Classification/06_Risk_Tier_Reporting",
    "risk_tier_packaging": "02_Problem2_Risk_Tier_Classification/07_Risk_Tier_Packaging",
}
_config_healed = False
for _key, _rel_path in _REQUIRED_PILLARS.items():
    if _key not in PILLAR_DIRS:
        PILLAR_DIRS[_key] = PROJECT_ROOT / _rel_path
        PROJECT_CONFIG["pillar_dirs"][_key] = str(PILLAR_DIRS[_key])
        _config_healed = True
        print(f"NOTE: '{_key}' was missing from project_config.json -- added automatically as {PILLAR_DIRS[_key]}")
if _config_healed:
    with open(CONFIG_PATH, "w", encoding="utf-8") as f:
        json.dump(PROJECT_CONFIG, f, indent=2)
    print("\u2705 project_config.json updated in place -- no need to re-run Notebook 01.")

MODEL_DEV_DIR = PILLAR_DIRS["model_development"]
MODELS_SUBDIR = MODEL_DEV_DIR / "models"
RISK_TIER_POLICY_DIR = PILLAR_DIRS["risk_tier_policy"]
RISK_TIER_MODELING_DIR = PILLAR_DIRS["risk_tier_modeling"]
RISK_TIER_MODELING_DIR.mkdir(parents=True, exist_ok=True)

RISK_TIER_POLICY_PATH = RISK_TIER_POLICY_DIR / "risk_tier_policy.json"
if not RISK_TIER_POLICY_PATH.exists():
    raise FileNotFoundError(
        f"{RISK_TIER_POLICY_PATH} not found.\nProblem 2's Notebook 20 has a hard dependency on Notebook 19's "
        f"policy -- fix: run 19_risk_tier_business_understanding.ipynb first."
    )
with open(RISK_TIER_POLICY_PATH, "r", encoding="utf-8") as f:
    RISK_TIER_POLICY = json.load(f)

TRAIN_SPLIT_ENG_PATH = Path(NB04_SUMMARY["output_files"]["train_split_engineered.csv"])
TEST_SPLIT_ENG_PATH = Path(NB04_SUMMARY["output_files"]["test_split_engineered.csv"])
MODEL_COMPARISON_PATH = MODEL_DEV_DIR / "model_comparison.csv"
CHAMPION_IMPORTANCE_PATH = MODEL_DEV_DIR / "champion_feature_importance.csv"
PREPROCESSING_PATH = MODELS_SUBDIR / "preprocessing_artifacts.joblib"

# --- Champion identification: same resilient pattern used throughout the platform --
NB05_SUMMARY = None
if NB05_SUMMARY_PATH.exists():
    with open(NB05_SUMMARY_PATH, "r", encoding="utf-8") as f:
        NB05_SUMMARY = json.load(f)
    CHAMPION_NAME = NB05_SUMMARY["champion_model"]
    _champion_source = f"{NB05_SUMMARY_PATH.name}"
elif MODEL_COMPARISON_PATH.exists():
    with open(MODEL_COMPARISON_PATH, "r", encoding="utf-8", newline="") as _f:
        _cmp_rows = list(csv.DictReader(_f))
    if not _cmp_rows or "model" not in _cmp_rows[0] or "holdout_amex_metric" not in _cmp_rows[0]:
        raise RuntimeError(f"{MODEL_COMPARISON_PATH} exists but is missing the expected 'model' / "
                            f"'holdout_amex_metric' columns -- cannot identify a champion from it. "
                            f"Fix: re-run 05_model_development.ipynb.")
    _champion_row = max(_cmp_rows, key=lambda r: float(r["holdout_amex_metric"]))
    CHAMPION_NAME = _champion_row["model"]
    _champion_source = f"{MODEL_COMPARISON_PATH.name} (fallback -- {NB05_SUMMARY_PATH.name} not found)"
else:
    raise FileNotFoundError(f"Neither {NB05_SUMMARY_PATH} nor {MODEL_COMPARISON_PATH} was found.\n"
                             f"Fix: run 05_model_development.ipynb first -- this notebook reuses its saved "
                             f"champion model and preprocessing artifacts.")

CHAMPION_MODEL_PATH = MODELS_SUBDIR / f"{CHAMPION_NAME}.joblib"

for _p in (TRAIN_SPLIT_ENG_PATH, TEST_SPLIT_ENG_PATH, CHAMPION_MODEL_PATH, PREPROCESSING_PATH,
           MODEL_COMPARISON_PATH, CHAMPION_IMPORTANCE_PATH):
    if not _p.exists():
        raise FileNotFoundError(f"Required file not found: {_p}\nFix: re-run 05_model_development.ipynb -- "
                                 f"this notebook reuses its saved champion model and outputs as-is.")

print(f"Champion model                : {CHAMPION_NAME}  (identified from: {_champion_source})")
print(f"Risk tier policy (Notebook 19) : primary_method='{RISK_TIER_POLICY['primary_method']}', "
      f"{RISK_TIER_POLICY['n_tiers']} tiers")
print(f"Risk tier modeling artifacts will be written under: {RISK_TIER_MODELING_DIR}")
print("\n\u2705 Section 1 complete.")


# =============================================================================
# SECTION 2: WARP HARDWARE CONFIGURATION, LIBRARY IMPORTS & ADAPTIVE RAM CEILING
# =============================================================================
_section("SECTION 2: WARP Hardware Configuration, Library Imports & Adaptive RAM Ceiling")

os.environ["POLARS_MAX_THREADS"] = str(WARP_THREAD_COUNT)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import gc

import logging
logger = logging.getLogger("amex_platform")
logger.setLevel(logging.INFO)
if not logger.handlers:
    _handler = logging.StreamHandler(sys.stdout)
    _handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S"))
    logger.addHandler(_handler)

missing = []
try:
    import polars as pl
except ImportError:
    missing.append("polars")
try:
    import numpy as np
except ImportError:
    missing.append("numpy")
try:
    import pandas as pd
except ImportError:
    missing.append("pandas")
try:
    import psutil
except ImportError:
    missing.append("psutil")
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except ImportError:
    missing.append("matplotlib")
try:
    import joblib
except ImportError:
    missing.append("joblib")
try:
    from scipy import stats as scipy_stats
except ImportError:
    missing.append("scipy")
try:
    from docx import Document
    from docx.shared import Inches, Pt
    from docx.enum.text import WD_ALIGN_PARAGRAPH
except ImportError:
    missing.append("python-docx")

if missing:
    raise ImportError(
        "Missing required package(s): " + ", ".join(missing) + "\n"
        "Fix: run this in a terminal, then re-run this cell:\n"
        f"    pip install {' '.join(missing)}"
    )

logger.info(f"Polars thread pool configured to {os.environ['POLARS_MAX_THREADS']} threads (95% cap, WARP 6.4, Concurrency)")


def _rss_gb() -> float:
    return psutil.Process().memory_info().rss / 1e9


_process_start_rss_gb = _rss_gb()
_live_vm = psutil.virtual_memory()
ADAPTIVE_RAM_FRACTION = _resource_limits.get("ram_fraction_cap", 0.90)
MAX_RAM_BYTES = int(_live_vm.available * ADAPTIVE_RAM_FRACTION)

print(f"Adaptive RAM ceiling (this run) : {MAX_RAM_BYTES / 1e9:.2f} GB")
print(f"\nProcess RSS at Section 2 start: {_process_start_rss_gb:.2f} GB")
print("\n\u2705 Section 2 complete.")


# =============================================================================
# SECTION 3: LOAD CHAMPION MODEL, PREPROCESSING ARTIFACTS & FEATURE IMPORTANCE
# =============================================================================
_section("SECTION 3: Load Champion Model, Preprocessing Artifacts & Feature Importance")

_t0 = time.time()
champion_model = joblib.load(CHAMPION_MODEL_PATH)
preprocessing_artifacts = joblib.load(PREPROCESSING_PATH)
print(f"Loaded champion model '{CHAMPION_NAME}' from {CHAMPION_MODEL_PATH} ({time.time() - _t0:.1f}s)")

label_encoders = preprocessing_artifacts["label_encoders"]
feature_medians = preprocessing_artifacts["feature_medians"]
scaler = preprocessing_artifacts["scaler"]
all_feature_cols = preprocessing_artifacts["all_feature_cols"]
categorical_encode_cols = preprocessing_artifacts["categorical_encode_cols"]
numeric_feature_cols = preprocessing_artifacts["numeric_feature_cols"]
champion_uses_scaled = CHAMPION_NAME == "logistic_regression"

champion_importance_df = pd.read_csv(CHAMPION_IMPORTANCE_PATH)
print(f"Feature columns loaded  : {len(all_feature_cols)} "
      f"({len(numeric_feature_cols)} numeric + {len(categorical_encode_cols)} categorical)")
print("\n\u2705 Section 3 complete.")


# =============================================================================
# SECTION 4: LOAD HOLDOUT ENGINEERED DATA, APPLY SAVED PREPROCESSING, SCORE
# =============================================================================
_section("SECTION 4: Load Holdout Engineered Data, Apply Saved Preprocessing, Score")

SPLIT_CSV_SCHEMA = {"customer_ID": pl.Utf8, "target": pl.Int8}
for _c in categorical_encode_cols:
    SPLIT_CSV_SCHEMA[_c] = pl.Utf8
for _c in numeric_feature_cols:
    SPLIT_CSV_SCHEMA[_c] = pl.Float32

_t0 = time.time()
train_pl = pl.read_csv(str(TRAIN_SPLIT_ENG_PATH), schema_overrides=SPLIT_CSV_SCHEMA)
holdout_pl = pl.read_csv(str(TEST_SPLIT_ENG_PATH), schema_overrides=SPLIT_CSV_SCHEMA)
print(f"Loaded test_split_engineered.csv : {holdout_pl.shape[0]:,} x {holdout_pl.shape[1]} (held-out, never trained on)")

_inf_clean_exprs = [
    pl.when(pl.col(c).is_infinite() | pl.col(c).is_nan()).then(None).otherwise(pl.col(c)).cast(pl.Float32).alias(c)
    for c in numeric_feature_cols
]
train_pl = train_pl.with_columns(_inf_clean_exprs)
holdout_pl = holdout_pl.with_columns(_inf_clean_exprs)

for c in categorical_encode_cols:
    train_pl = train_pl.with_columns(pl.col(c).cast(pl.Utf8).fill_null("__missing__").alias(c))
    holdout_pl = holdout_pl.with_columns(pl.col(c).cast(pl.Utf8).fill_null("__missing__").alias(c))
    _mapping = {cat: i for i, cat in enumerate(label_encoders[c]["classes"])}
    train_pl = train_pl.with_columns(pl.col(c).replace_strict(_mapping, default=-1, return_dtype=pl.Int32).alias(c))
    holdout_pl = holdout_pl.with_columns(pl.col(c).replace_strict(_mapping, default=-1, return_dtype=pl.Int32).alias(c))

_impute_exprs = [pl.col(c).fill_null(feature_medians[c]) for c in numeric_feature_cols]
train_pl = train_pl.with_columns(_impute_exprs)
holdout_pl = holdout_pl.with_columns(_impute_exprs)

X_holdout = holdout_pl.select(all_feature_cols).to_numpy().astype(np.float32, copy=False)
y_holdout = holdout_pl.get_column("target").to_numpy().astype(np.int64, copy=False)
holdout_customer_ids = holdout_pl.get_column("customer_ID").to_numpy()

_train_mean = scaler["mean"]
_train_std = scaler["std"]
X_holdout_scaled = (X_holdout - _train_mean) / _train_std

Xc_holdout = X_holdout_scaled if champion_uses_scaled else X_holdout
PD_HOLDOUT = champion_model.predict_proba(Xc_holdout)[:, 1]

del train_pl, holdout_pl
gc.collect()
print(f"Scored {X_holdout.shape[0]:,} holdout customers with the real champion model ({time.time() - _t0:.1f}s total)")
print(f"Process RSS now: {_rss_gb():.2f} GB (of {MAX_RAM_BYTES / 1e9:.2f} GB adaptive ceiling)")
print("\n\u2705 Section 4 complete.")


# =============================================================================
# SECTION 5: COMPUTE BOTH TIER-BUCKETING METHODS FROM NOTEBOOK 19'S REAL POLICY
# =============================================================================
_section("SECTION 5: Compute Both Tier-Bucketing Methods From Notebook 19's Real Policy")

TIER_ORDER = RISK_TIER_POLICY["tier_order"]  # e.g. ["Prime", "Near-Prime", "Subprime", "High Risk"] -- ascending risk
N_TIERS = RISK_TIER_POLICY["n_tiers"]

# --- Business-rule method: fixed PD thresholds from the real policy JSON ---
_br_thresholds = sorted(RISK_TIER_POLICY["bucketing_methods"]["business_rule"]["pd_thresholds"],
                         key=lambda b: b["tier_order"])


def _assign_business_rule(pd_value: float) -> str:
    for band in _br_thresholds:
        if band["pd_lower"] <= pd_value < band["pd_upper"]:
            return band["risk_tier"]
    return _br_thresholds[-1]["risk_tier"]


risk_tier_business_rule = np.array([_assign_business_rule(p) for p in PD_HOLDOUT])

# --- Quantile method: real, live-computed empirical quantile cutpoints of THIS
#     run's actual holdout PD distribution -- not a fixed value, recomputed
#     every run so it always reflects the real, current scored population. ---
_q_cutpoints = RISK_TIER_POLICY["bucketing_methods"]["quantile"]["quantile_cutpoints"]  # e.g. [0.25, 0.5, 0.75]
_q_edges = [float(np.quantile(PD_HOLDOUT, q)) for q in _q_cutpoints]
_q_bin_edges = [-np.inf] + _q_edges + [np.inf]
_q_bin_indices = np.digitize(PD_HOLDOUT, _q_edges, right=False)  # 0..N_TIERS-1
risk_tier_quantile = np.array([TIER_ORDER[i] for i in _q_bin_indices])

print(f"Business-rule thresholds (real policy, Notebook 19): "
      f"{[(b['risk_tier'], b['pd_lower'], b['pd_upper']) for b in _br_thresholds]}")
print(f"Quantile cutpoints (real, computed live from this run's PD_HOLDOUT): "
      f"{[round(e, 4) for e in _q_edges]}")

PRIMARY_METHOD = RISK_TIER_POLICY["primary_method"]
risk_tier_primary = risk_tier_business_rule if PRIMARY_METHOD == "business_rule" else risk_tier_quantile
print(f"\nPrimary method (per Notebook 19's policy): {PRIMARY_METHOD}")
print("\n\u2705 Section 5 complete.")


# =============================================================================
# SECTION 6: PER-TIER PERFORMANCE METRICS -- BOTH METHODS, MEASURED HONESTLY
# =============================================================================
_section("SECTION 6: Per-Tier Performance Metrics -- Both Methods, Measured Honestly")

KPI_TARGETS = RISK_TIER_POLICY["kpi_targets"]


def _tier_metrics(tier_assignments, method_name):
    rows = []
    for tier in TIER_ORDER:
        _mask = tier_assignments == tier
        _n = int(_mask.sum())
        _bad_rate = float(y_holdout[_mask].mean()) if _n > 0 else float("nan")
        _mean_pd = float(PD_HOLDOUT[_mask].mean()) if _n > 0 else float("nan")
        rows.append({
            "method": method_name, "risk_tier": tier, "n_accounts": _n,
            "population_pct": round(100.0 * _n / len(tier_assignments), 2),
            "actual_bad_rate_pct": round(100.0 * _bad_rate, 3) if _n > 0 else None,
            "mean_predicted_pd": round(_mean_pd, 4) if _n > 0 else None,
            "calibration_gap": round(_mean_pd - _bad_rate, 4) if _n > 0 else None,
        })
    return rows


def _kpi_compliance(tier_rows, method_name):
    _bad_rates = [r["actual_bad_rate_pct"] for r in tier_rows]
    _valid = all(v is not None for v in _bad_rates)
    _monotonic = _valid and all(_bad_rates[i] < _bad_rates[i + 1] for i in range(len(_bad_rates) - 1))
    _ratio = (_bad_rates[-1] / _bad_rates[0]) if (_valid and _bad_rates[0] > 0) else None
    _ratio_ok = (_ratio is not None) and (_ratio >= KPI_TARGETS["min_bad_rate_ratio_top_to_bottom_tier"])
    _min_pop_pct = min(r["population_pct"] for r in tier_rows)
    _pop_ok = _min_pop_pct >= KPI_TARGETS["min_tier_population_pct"]
    return {
        "method": method_name,
        "monotonicity_pass": bool(_monotonic) if KPI_TARGETS["require_strict_monotonicity"] else True,
        "monotonicity_measured": bool(_monotonic),
        "bad_rate_ratio_top_to_bottom": round(_ratio, 2) if _ratio is not None else None,
        "bad_rate_ratio_target": KPI_TARGETS["min_bad_rate_ratio_top_to_bottom_tier"],
        "bad_rate_ratio_pass": bool(_ratio_ok),
        "min_tier_population_pct_measured": round(_min_pop_pct, 2),
        "min_tier_population_pct_target": KPI_TARGETS["min_tier_population_pct"],
        "population_balance_pass": bool(_pop_ok),
    }


business_rule_tier_rows = _tier_metrics(risk_tier_business_rule, "business_rule")
quantile_tier_rows = _tier_metrics(risk_tier_quantile, "quantile")
tier_performance_df = pd.DataFrame(business_rule_tier_rows + quantile_tier_rows)

business_rule_kpi = _kpi_compliance(business_rule_tier_rows, "business_rule")
quantile_kpi = _kpi_compliance(quantile_tier_rows, "quantile")

print(tier_performance_df.to_string(index=False))
print(f"\nBusiness-rule KPI compliance : {business_rule_kpi}")
print(f"Quantile KPI compliance      : {quantile_kpi}")
print("\nNote: a KPI target not met is a real, measured outcome on this holdout population -- reported honestly "
      "below and in the Word report, never silently marked as passing.")
print("\n\u2705 Section 6 complete.")


# =============================================================================
# SECTION 7: SPLIT-HALF POPULATION STABILITY (PSI) -- PRIMARY METHOD
# =============================================================================
_section("SECTION 7: Split-Half Population Stability (PSI) -- Primary Method")

# --- Honest framing: this dataset carries no real time-ordering signal usable
#     for genuine drift measurement at this notebook's scope (Notebook 23
#     revisits temporal monitoring using Notebook 12's simulated windows).
#     Here, a deterministic random split-half of the SAME holdout population
#     is used as a real, reproducible stability proxy: it tests whether tier
#     composition is sensitive to which random subset of accounts is scored --
#     a genuine, live-computed statistic, just not a genuine time-drift test. ---
_rng = np.random.default_rng(RANDOM_SEED)
_perm = _rng.permutation(len(risk_tier_primary))
_half = len(_perm) // 2
_half_a_tiers = risk_tier_primary[_perm[:_half]]
_half_b_tiers = risk_tier_primary[_perm[_half:]]


def _tier_pct_dist(tier_assignments):
    _counts = pd.Series(tier_assignments).value_counts().reindex(TIER_ORDER).fillna(0)
    return (_counts / _counts.sum()).clip(lower=1e-4)


_dist_a = _tier_pct_dist(_half_a_tiers)
_dist_b = _tier_pct_dist(_half_b_tiers)
PSI_SPLIT_HALF = float(((_dist_a - _dist_b) * np.log(_dist_a / _dist_b)).sum())
PSI_PASS = PSI_SPLIT_HALF <= KPI_TARGETS["max_tier_population_psi_split_half"]

print(f"Split-half A population : {dict((_dist_a * 100).round(2))}")
print(f"Split-half B population : {dict((_dist_b * 100).round(2))}")
print(f"PSI (split-half, primary method '{PRIMARY_METHOD}') : {PSI_SPLIT_HALF:.4f} "
      f"(target <= {KPI_TARGETS['max_tier_population_psi_split_half']})")
_psi_status_label = "\u2705 PASS" if PSI_PASS else "\u274c FAIL"
print(f"Stability check: {_psi_status_label}")
print("\n\u2705 Section 7 complete.")


# =============================================================================
# SECTION 8: STATISTICAL SIGNIFICANCE -- CHI-SQUARE TEST OF TIER VS. DEFAULT
# =============================================================================
_section("SECTION 8: Statistical Significance -- Chi-Square Test of Tier vs. Default")

_contingency = pd.crosstab(pd.Series(risk_tier_primary, name="risk_tier"), pd.Series(y_holdout, name="actual_default"))
_contingency = _contingency.reindex(TIER_ORDER)
_chi2, _chi2_p, _chi2_dof, _ = scipy_stats.chi2_contingency(_contingency.values)
CHI2_STATISTIC = float(_chi2)
CHI2_P_VALUE = float(_chi2_p)
CHI2_SIGNIFICANT = CHI2_P_VALUE < 0.001

print(_contingency.to_string())
print(f"\nChi-square statistic : {CHI2_STATISTIC:.2f}  (dof={_chi2_dof})")
print(f"p-value               : {CHI2_P_VALUE:.3e}")
print(f"Real risk tier vs. real actual default association is "
      f"{'statistically significant (p < 0.001)' if CHI2_SIGNIFICANT else 'NOT statistically significant at the 0.001 level'}.")
print("\n\u2705 Section 8 complete.")


# =============================================================================
# SECTION 9: FINAL RISK-TIER ASSIGNMENTS TABLE (BOTH METHODS + PRIMARY)
# =============================================================================
_section("SECTION 9: Final Risk-Tier Assignments Table (Both Methods + Primary)")

risk_tier_assignments_df = pd.DataFrame({
    "customer_ID": holdout_customer_ids,
    "predicted_pd": np.round(PD_HOLDOUT, 6),
    "actual_default": y_holdout,
    "risk_tier_quantile": risk_tier_quantile,
    "risk_tier_business_rule": risk_tier_business_rule,
    "risk_tier_primary": risk_tier_primary,
    "primary_method": PRIMARY_METHOD,
})

assignments_path = RISK_TIER_MODELING_DIR / "risk_tier_assignments.csv"
risk_tier_assignments_df.to_csv(assignments_path, index=False)
print(f"risk_tier_assignments.csv: {risk_tier_assignments_df.shape[0]:,} rows x {risk_tier_assignments_df.shape[1]} columns")
print(f"\u2705 Saved -> {assignments_path}  ({assignments_path.stat().st_size / 1e6:.2f} MB)")

tier_performance_path = RISK_TIER_MODELING_DIR / "risk_tier_performance_by_method.csv"
tier_performance_df.to_csv(tier_performance_path, index=False)
print(f"\u2705 Saved -> {tier_performance_path}")
print("\n\u2705 Section 9 complete.")


# =============================================================================
# SECTION 10: CHARTS
# =============================================================================
_section("SECTION 10: Charts")

VIZ = {"surface": "#fcfcfb", "text_primary": "#0b0b0b", "text_secondary": "#52514e", "grid": "#e3e2dd",
       "cat_blue": "#2a78d6", "cat_red": "#e34948", "cat_green": "#3a9e5f", "cat_amber": "#d69a2a"}
PROBLEM_NAME = "Phase 1 \u00b7 Problem 2 -- Risk Tier Classification"


def _style_axes(ax):
    ax.set_facecolor(VIZ["surface"]); ax.figure.set_facecolor(VIZ["surface"])
    ax.grid(axis="y", color=VIZ["grid"], linewidth=0.8, zorder=0); ax.set_axisbelow(True)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
    for spine in ("left", "bottom"):
        ax.spines[spine].set_color(VIZ["grid"])
    ax.tick_params(colors=VIZ["text_secondary"], labelsize=9)
    ax.title.set_color(VIZ["text_primary"])
    ax.xaxis.label.set_color(VIZ["text_secondary"]); ax.yaxis.label.set_color(VIZ["text_secondary"])


# Chart 1: actual bad rate by tier -- both methods, grouped bars
_br_rates = [r["actual_bad_rate_pct"] for r in business_rule_tier_rows]
_q_rates = [r["actual_bad_rate_pct"] for r in quantile_tier_rows]
_x = np.arange(len(TIER_ORDER)); _w = 0.35
fig, ax = plt.subplots(figsize=(8.5, 5.5), dpi=150)
ax.bar(_x - _w / 2, _br_rates, _w, label="Business-rule", color=VIZ["cat_blue"], zorder=3)
ax.bar(_x + _w / 2, _q_rates, _w, label="Quantile", color=VIZ["cat_amber"], zorder=3)
ax.set_xticks(_x); ax.set_xticklabels(TIER_ORDER)
_style_axes(ax)
ax.set_ylabel("Actual default rate (%, holdout, real)")
ax.set_title(f"{PROBLEM_NAME}\nActual Bad Rate by Tier -- Both Methods (Real, Measured)", fontsize=11)
ax.legend(frameon=False, fontsize=9)
fig.tight_layout()
chart1_path = RISK_TIER_MODELING_DIR / "bad_rate_by_tier_both_methods_chart.png"
fig.savefig(chart1_path, dpi=150, facecolor=VIZ["surface"])
plt.show(); plt.close(fig)
print(f"\u2705 Saved -> {chart1_path}")

# Chart 2: population distribution -- both methods
_br_pop = [r["population_pct"] for r in business_rule_tier_rows]
_q_pop = [r["population_pct"] for r in quantile_tier_rows]
fig, ax = plt.subplots(figsize=(8.5, 5.5), dpi=150)
ax.bar(_x - _w / 2, _br_pop, _w, label="Business-rule", color=VIZ["cat_blue"], zorder=3)
ax.bar(_x + _w / 2, _q_pop, _w, label="Quantile", color=VIZ["cat_amber"], zorder=3)
ax.set_xticks(_x); ax.set_xticklabels(TIER_ORDER)
_style_axes(ax)
ax.set_ylabel("Population (%, holdout, real)")
ax.set_title(f"{PROBLEM_NAME}\nPopulation Distribution by Tier -- Both Methods (Real, Measured)", fontsize=11)
ax.legend(frameon=False, fontsize=9)
fig.tight_layout()
chart2_path = RISK_TIER_MODELING_DIR / "population_by_tier_both_methods_chart.png"
fig.savefig(chart2_path, dpi=150, facecolor=VIZ["surface"])
plt.show(); plt.close(fig)
print(f"\u2705 Saved -> {chart2_path}")

# Chart 3: predicted PD distribution, colored by primary tier
fig, ax = plt.subplots(figsize=(8.5, 5.5), dpi=150)
_tier_colors = {TIER_ORDER[0]: VIZ["cat_green"], TIER_ORDER[1]: VIZ["cat_blue"],
                TIER_ORDER[2]: VIZ["cat_amber"], TIER_ORDER[-1]: VIZ["cat_red"]}
for tier in TIER_ORDER:
    _vals = PD_HOLDOUT[risk_tier_primary == tier]
    if len(_vals) > 0:
        ax.hist(_vals, bins=30, alpha=0.75, label=tier, color=_tier_colors.get(tier, VIZ["cat_blue"]), zorder=3)
_style_axes(ax)
ax.set_xlabel("Predicted PD"); ax.set_ylabel("Number of accounts")
ax.set_title(f"{PROBLEM_NAME}\nPredicted PD Distribution by Primary Tier ('{PRIMARY_METHOD}', Real)", fontsize=11)
ax.legend(frameon=False, fontsize=9)
fig.tight_layout()
chart3_path = RISK_TIER_MODELING_DIR / "pd_distribution_by_tier_chart.png"
fig.savefig(chart3_path, dpi=150, facecolor=VIZ["surface"])
plt.show(); plt.close(fig)
print(f"\u2705 Saved -> {chart3_path}")

# Chart 4: calibration -- mean predicted PD vs. actual bad rate, primary method
_primary_rows = business_rule_tier_rows if PRIMARY_METHOD == "business_rule" else quantile_tier_rows
_mean_pd_vals = [r["mean_predicted_pd"] * 100 for r in _primary_rows]
_bad_rate_vals = [r["actual_bad_rate_pct"] for r in _primary_rows]
fig, ax = plt.subplots(figsize=(8.5, 5.5), dpi=150)
ax.plot(_x, _mean_pd_vals, marker="o", label="Mean predicted PD (%)", color=VIZ["cat_blue"], linewidth=2, zorder=3)
ax.plot(_x, _bad_rate_vals, marker="s", label="Actual bad rate (%)", color=VIZ["cat_red"], linewidth=2, zorder=3)
ax.set_xticks(_x); ax.set_xticklabels(TIER_ORDER)
_style_axes(ax)
ax.set_ylabel("Rate (%)")
ax.set_title(f"{PROBLEM_NAME}\nCalibration -- Predicted vs. Actual by Tier (Primary Method, Real)", fontsize=11)
ax.legend(frameon=False, fontsize=9)
fig.tight_layout()
chart4_path = RISK_TIER_MODELING_DIR / "calibration_by_tier_chart.png"
fig.savefig(chart4_path, dpi=150, facecolor=VIZ["surface"])
plt.show(); plt.close(fig)
print(f"\u2705 Saved -> {chart4_path}")

print("\n\u2705 Section 10 complete.")


# =============================================================================
# SECTION 11: WORD REPORT -- RISK_TIER_MODEL_DEVELOPMENT_REPORT.DOCX
# =============================================================================
_section("SECTION 11: Word Report -- Risk_Tier_Model_Development_Report.docx")

doc = Document()
doc.add_heading("AMEX Enterprise Credit Risk Platform", level=0)
doc.add_paragraph(f"Phase 1, Problem 2: Risk Tier Classification -- Model Development Report")
doc.add_paragraph(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

doc.add_heading("1. Methodology", level=1)
doc.add_paragraph(
    f"Problem 1's real champion model ({CHAMPION_NAME}, holdout AUC {RISK_TIER_POLICY['champion_holdout_auc']}, "
    f"holdout AMEX metric {RISK_TIER_POLICY['champion_holdout_amex_metric']}) scores the real, held-out test "
    f"split ({len(risk_tier_assignments_df):,} customers, never trained on -- the same split Notebook 05 and "
    f"Notebook 13 use). Two tier-bucketing methods, both defined in Notebook 19's policy, are computed against "
    f"these real PD scores and compared."
)

doc.add_heading("2. Per-Tier Performance -- Both Methods (Real, Measured)", level=1)
_t = doc.add_table(rows=1, cols=len(tier_performance_df.columns))
_t.style = "Light Grid Accent 1"
for _i, _col in enumerate(tier_performance_df.columns):
    _t.rows[0].cells[_i].text = _col.replace("_", " ").title()
for _, _row in tier_performance_df.iterrows():
    _cells = _t.add_row().cells
    for _i, _col in enumerate(tier_performance_df.columns):
        _cells[_i].text = str(_row[_col])

doc.add_heading("3. KPI Compliance -- Honest Pass/Fail", level=1)
doc.add_paragraph(
    "A KPI target not met below is a real, measured outcome on this holdout population -- reported as-is, "
    "never adjusted to appear compliant."
)
for _kpi, _label in [(business_rule_kpi, "Business-Rule Method"), (quantile_kpi, "Quantile Method")]:
    doc.add_heading(_label, level=2)
    for _k, _v in _kpi.items():
        if _k == "method":
            continue
        _mark = ""
        if isinstance(_v, bool):
            _mark = "PASS" if _v else "FAIL"
        doc.add_paragraph(f"{_k.replace('_', ' ').title()}: {_v}" + (f"  [{_mark}]" if _mark else ""))

doc.add_heading("4. Population Stability (Split-Half PSI, Primary Method)", level=1)
doc.add_paragraph(
    f"PSI = {PSI_SPLIT_HALF:.4f} (target <= {KPI_TARGETS['max_tier_population_psi_split_half']}) -- "
    f"{'PASS' if PSI_PASS else 'FAIL'}. Honest scope note: this is a random split-half stability proxy on the "
    f"single available holdout population, not a genuine time-based drift measurement (Notebook 23 covers "
    f"simulated temporal monitoring windows, reusing Notebook 12's real pattern)."
)

doc.add_heading("5. Statistical Significance", level=1)
doc.add_paragraph(
    f"Chi-square test of risk tier vs. actual default: statistic={CHI2_STATISTIC:.2f}, dof={_chi2_dof}, "
    f"p-value={CHI2_P_VALUE:.3e} -- "
    f"{'statistically significant (p < 0.001)' if CHI2_SIGNIFICANT else 'not statistically significant at the 0.001 level'}."
)

doc.add_heading("6. Selected Primary Method", level=1)
doc.add_paragraph(
    f"Per Notebook 19's policy, '{PRIMARY_METHOD}' is the primary method used downstream (Notebooks 21-25). "
    f"Both methods' assignments are retained in risk_tier_assignments.csv for transparency and comparison."
)

doc.add_heading("7. Charts", level=1)
for _cp, _cap in [(chart1_path, "Actual bad rate by tier"), (chart2_path, "Population distribution by tier"),
                   (chart3_path, "Predicted PD distribution by tier"), (chart4_path, "Calibration by tier")]:
    doc.add_picture(str(_cp), width=Inches(6.0))
    _p = doc.add_paragraph(_cap); _p.alignment = WD_ALIGN_PARAGRAPH.CENTER

report_path = RISK_TIER_MODELING_DIR / "Risk_Tier_Model_Development_Report.docx"
doc.save(report_path)
print(f"\u2705 Saved -> {report_path}")
print("\n\u2705 Section 11 complete.")


# =============================================================================
# SECTION 12: VERIFICATION -- STRUCTURAL INTEGRITY (FATAL) & KPI STATUS (REPORTED)
# =============================================================================
_section("SECTION 12: Verification")

_checks_passed = True


def _check(label, condition, detail=""):
    global _checks_passed
    if condition:
        print(f"\u2705 {label}")
    else:
        _checks_passed = False
        print(f"\u274c {label}  {detail}")


# --- Structural integrity checks: these are fatal. A failure here means the
#     notebook itself malfunctioned, not that a real-world KPI came up short. ---
_check("Assignments row count matches the holdout population", len(risk_tier_assignments_df) == X_holdout.shape[0],
       f"({len(risk_tier_assignments_df)} vs {X_holdout.shape[0]})")
_check("Every account has both a quantile and business-rule tier assigned",
       risk_tier_assignments_df["risk_tier_quantile"].notna().all()
       and risk_tier_assignments_df["risk_tier_business_rule"].notna().all())
_check("Quantile method produced exactly N_TIERS distinct tiers",
       risk_tier_assignments_df["risk_tier_quantile"].nunique() == N_TIERS,
       f"({risk_tier_assignments_df['risk_tier_quantile'].nunique()} vs {N_TIERS})")
_check("Tier performance table covers both methods x N_TIERS rows",
       len(tier_performance_df) == 2 * N_TIERS, f"({len(tier_performance_df)})")
_check("Chi-square contingency table has no negative or NaN cells",
       bool((_contingency.values >= 0).all()) and not np.isnan(_contingency.values).any())

_expected_files = [assignments_path, tier_performance_path, chart1_path, chart2_path, chart3_path, chart4_path,
                    report_path]
for fp in _expected_files:
    _check(f"{fp.name} exists and is non-empty", fp.exists() and fp.stat().st_size > 0)

if not _checks_passed:
    raise RuntimeError("One or more Notebook 20 STRUCTURAL verification checks failed. See \u274c lines above.")

print("\nAll Notebook 20 structural checks passed.")

# --- KPI compliance status: reported honestly, NEVER fatal. A real KPI miss on
#     real data is a finding for the report, not a bug to hide or crash on. ---
_primary_kpi = business_rule_kpi if PRIMARY_METHOD == "business_rule" else quantile_kpi
_primary_kpi_all_pass = (_primary_kpi["monotonicity_pass"] and _primary_kpi["bad_rate_ratio_pass"]
                          and _primary_kpi["population_balance_pass"] and PSI_PASS)
_primary_kpi_status_label = ("\u2705 PASS" if _primary_kpi_all_pass
                              else "\u274c FAIL -- see report for detail (real measured outcome, not a notebook defect)")
print(f"\nPrimary method ('{PRIMARY_METHOD}') full KPI compliance: {_primary_kpi_status_label}")
print("\n\u2705 Section 12 complete.")


# =============================================================================
# SECTION 13: RESOURCE / PERFORMANCE REPORT
# =============================================================================
_section("SECTION 13: Resource / Performance Report")

_final_rss_gb = _rss_gb()
performance_report = {
    "warp_thread_count_configured": WARP_THREAD_COUNT,
    "adaptive_ram_ceiling_gb": round(MAX_RAM_BYTES / 1e9, 2),
    "process_rss_at_start_gb": round(_process_start_rss_gb, 2),
    "process_rss_at_end_gb": round(_final_rss_gb, 2),
    "holdout_rows_scored": int(X_holdout.shape[0]),
}
performance_report_path = ARTIFACTS_DIR / "notebook_20_performance_report.json"
with open(performance_report_path, "w", encoding="utf-8") as f:
    json.dump(performance_report, f, indent=2)
print(f"Process RSS: {_process_start_rss_gb:.2f} GB (start) -> {_final_rss_gb:.2f} GB (end)")
print(f"\u2705 Saved -> {performance_report_path}")
print("\n\u2705 Section 13 complete.")


# =============================================================================
# SECTION 14: WRITE NOTEBOOK 20 SUMMARY ARTIFACT (for Notebook 24's rollup)
# =============================================================================
_section("SECTION 14: Write Notebook 20 Summary Artifact")

notebook_20_summary = {
    "notebook": "20_risk_tier_model_development",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "problem_number": 2,
    "problem_name": "Risk Tier Classification",
    "champion_model": CHAMPION_NAME,
    "primary_method": PRIMARY_METHOD,
    "n_tiers": N_TIERS,
    "holdout_rows_scored": int(X_holdout.shape[0]),
    "tier_performance": business_rule_tier_rows + quantile_tier_rows,
    "kpi_compliance": {"business_rule": business_rule_kpi, "quantile": quantile_kpi},
    "primary_method_full_kpi_pass": bool(_primary_kpi_all_pass),
    "split_half_psi": round(PSI_SPLIT_HALF, 4),
    "split_half_psi_pass": bool(PSI_PASS),
    "chi_square_statistic": CHI2_STATISTIC,
    "chi_square_p_value": CHI2_P_VALUE,
    "chi_square_significant": bool(CHI2_SIGNIFICANT),
    "output_files": {p.name: str(p) for p in _expected_files + [performance_report_path]},
}
nb20_summary_path = ARTIFACTS_DIR / "notebook_20_summary.json"
with open(nb20_summary_path, "w", encoding="utf-8") as f:
    json.dump(notebook_20_summary, f, indent=2)
print(f"\u2705 Saved -> {nb20_summary_path}")
print("\n\u2705 Section 14 complete.")


# =============================================================================
# SECTION 15: COMPLETION SUMMARY
# =============================================================================
_section("SECTION 15: Notebook 20 Complete -- Handoff to Notebook 21")

print("NOTEBOOK 20: RISK TIER MODEL DEVELOPMENT -- COMPLETE")
print(f"  Champion model                    : {CHAMPION_NAME}")
print(f"  Primary method                    : {PRIMARY_METHOD}")
print(f"  Primary method full KPI compliance : {'PASS' if _primary_kpi_all_pass else 'FAIL (see report)'}")
print(f"  Split-half PSI                    : {PSI_SPLIT_HALF:.4f} ({'PASS' if PSI_PASS else 'FAIL'})")
print(f"  Chi-square p-value                : {CHI2_P_VALUE:.3e}")
print(f"  Files produced                    : {len(_expected_files) + 2}")
for _p in _expected_files + [performance_report_path, nb20_summary_path]:
    print(f"    - {_p.name}")
print(f"  Next notebook                     : 21_risk_tier_validation.ipynb")
print("\n\u2705 Ready to proceed.")
